# 01 - Data Acquisition

Download and merge marine fish DNA barcode data from multiple sources:
- **BOLD Systems** — Australian + Indo-Pacific fish COI barcodes
- **Mare-MAGE** — Curated global fish barcode database (COI + 12S)
- **NCBI GenBank** — Gap-filling for missing species

**Output**: `data/raw/` directory with downloaded files + `data/raw/merged_barcodes.csv`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kluless13/marinemamba/blob/main/notebooks/01_data_acquisition.ipynb)

In [ ]:
# === COLAB SETUP ===
# Uncomment if running in Google Colab
# !pip install biopython requests tqdm pandas
# !git clone https://github.com/kluless13/marinemamba.git
# %cd marinemamba

In [ ]:
import os
import requests
import pandas as pd
import json
from io import StringIO
from pathlib import Path
from tqdm import tqdm
import time

# Create data directories
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Saving raw data to: {RAW_DIR.resolve()}")

## 1. BOLD Systems — Australian Fish COI Barcodes

The Barcode of Life Data System has 327K+ Actinopterygii specimens globally, ~14K from Australia.
We query the public API for combined data (sequences + taxonomy).

In [ ]:
def download_bold_fish(taxon="Actinopterygii", geo=None, marker="COI-5P", output_path=None):
    """Download fish barcodes from BOLD Systems API."""
    base_url = "https://v4.boldsystems.org/index.php/API_Public/combined"
    params = {
        "taxon": taxon,
        "marker": marker,
        "format": "tsv"
    }
    if geo:
        params["geo"] = geo
    
    print(f"Downloading BOLD: taxon={taxon}, geo={geo}, marker={marker}")
    print(f"URL: {base_url}?{'&'.join(f'{k}={v}' for k,v in params.items())}")
    
    try:
        response = requests.get(base_url, params=params, timeout=300)
        response.raise_for_status()
        
        if output_path:
            with open(output_path, 'w') as f:
                f.write(response.text)
            print(f"Saved to {output_path}")
        
        # Parse TSV — BOLD returns messy data, need error handling
        df = pd.read_csv(
            StringIO(response.text),
            sep='\t',
            low_memory=False,
            on_bad_lines='skip',    # Skip malformed rows
            quoting=3,              # QUOTE_NONE — BOLD data has unescaped quotes
            engine='python'
        )
        
        # Drop rows without nucleotide data
        if 'nucleotides' in df.columns:
            df = df.dropna(subset=['nucleotides'])
            df = df[df['nucleotides'].str.len() > 0]
        
        print(f"Downloaded {len(df)} records, {df['species_name'].nunique() if 'species_name' in df.columns else '?'} species")
        return df
        
    except requests.exceptions.RequestException as e:
        print(f"BOLD API error: {e}")
        print("Note: BOLD API can be intermittent. Try again or download manually.")
        return pd.DataFrame()

In [ ]:
# Download Australian fish COI
bold_aus = download_bold_fish(
    taxon="Actinopterygii",
    geo="Australia",
    marker="COI-5P",
    output_path=RAW_DIR / "bold_aus_actinopterygii_coi.tsv"
)

In [ ]:
# Download broader Indo-Pacific families (key GBR families)
# Split into family-level queries to avoid timeout
gbr_families = [
    "Pomacentridae",   # damselfishes
    "Labridae",        # wrasses
    "Chaetodontidae",  # butterflyfishes
    "Acanthuridae",    # surgeonfishes
    "Scaridae",        # parrotfishes
    "Serranidae",      # groupers
    "Gobiidae",        # gobies
    "Blenniidae",      # blennies
    "Apogonidae",      # cardinalfishes
    "Lutjanidae",      # snappers
    "Mullidae",        # goatfishes
    "Pomacanthidae",   # angelfishes
    "Holocentridae",   # squirrelfishes
    "Muraenidae",      # moray eels
    "Syngnathidae",    # seahorses/pipefish
    "Scorpaenidae",    # scorpionfishes
    "Balistidae",      # triggerfishes
    "Tetraodontidae",  # pufferfishes
    "Lethrinidae",     # emperors
    "Siganidae",       # rabbitfishes
]

bold_families = []
for family in tqdm(gbr_families, desc="Downloading BOLD families"):
    df = download_bold_fish(
        taxon=family,
        marker="COI-5P",
        output_path=RAW_DIR / f"bold_{family.lower()}_coi.tsv"
    )
    if len(df) > 0:
        bold_families.append(df)
    time.sleep(2)  # Rate limiting

bold_global = pd.concat(bold_families, ignore_index=True) if bold_families else pd.DataFrame()
print(f"\nTotal global family records: {len(bold_global)}")

## 2. NCBI GenBank — Gap-filling

Use Biopython E-utilities to search for fish COI barcodes not in BOLD.

In [ ]:
from Bio import Entrez, SeqIO

# TODO: Set your email and API key
Entrez.email = "YOUR_EMAIL@example.com"  # Required by NCBI
Entrez.api_key = None  # Optional but recommended (10 req/sec vs 3)

def search_genbank_fish_coi(query=None, retmax=10000):
    """Search GenBank for fish COI barcode sequences."""
    if query is None:
        query = (
            '("COI"[Gene] OR "cox1"[Gene] OR "cytochrome c oxidase subunit 1"[Title]) '
            'AND "Actinopterygii"[Organism] '
            'AND 600:700[Sequence Length] '
            'AND "barcode"[All Fields]'
        )
    
    print(f"Searching GenBank: {query}")
    handle = Entrez.esearch(db="nuccore", term=query, retmax=0, usehistory="y")
    results = Entrez.read(handle)
    handle.close()
    
    total = int(results["Count"])
    print(f"Found {total} sequences")
    
    return {
        "count": total,
        "webenv": results["WebEnv"],
        "query_key": results["QueryKey"]
    }

def fetch_genbank_batch(search_result, batch_size=500, max_records=50000):
    """Fetch sequences in batches from GenBank."""
    records = []
    total = min(search_result["count"], max_records)
    
    for start in tqdm(range(0, total, batch_size), desc="Fetching GenBank"):
        try:
            handle = Entrez.efetch(
                db="nuccore",
                rettype="fasta",
                retmode="text",
                retstart=start,
                retmax=batch_size,
                webenv=search_result["webenv"],
                query_key=search_result["query_key"]
            )
            batch = list(SeqIO.parse(handle, "fasta"))
            records.extend(batch)
            handle.close()
            time.sleep(0.5)  # Rate limiting
        except Exception as e:
            print(f"Error at batch {start}: {e}")
            time.sleep(5)
            continue
    
    print(f"Fetched {len(records)} sequences from GenBank")
    return records

In [ ]:
# Search and download
search_result = search_genbank_fish_coi()

# Fetch (cap at 50K to be reasonable for first pass)
genbank_records = fetch_genbank_batch(search_result, max_records=50000)

# Save as FASTA
genbank_fasta = RAW_DIR / "genbank_fish_coi.fasta"
SeqIO.write(genbank_records, genbank_fasta, "fasta")
print(f"Saved {len(genbank_records)} sequences to {genbank_fasta}")

## 3. Merge All Sources

Standardize all data into a single CSV with BarcodeMamba-compatible columns.

In [ ]:
def standardize_bold(df):
    """Standardize BOLD data to common format."""
    cols = {
        "processid": "processid",
        "nucleotides": "nucleotides",
        "species_name": "species_name",
        "genus_name": "genus_name",
        "family_name": "family_name",
        "order_name": "order_name",
        "bin_uri": "bin_uri"
    }
    available = {k: v for k, v in cols.items() if k in df.columns}
    result = df[list(available.keys())].rename(columns=available).copy()
    result["source"] = "BOLD"
    return result

def standardize_genbank(records):
    """Standardize GenBank FASTA records to common format."""
    rows = []
    for rec in records:
        # Parse species from description (format: "ACC_ID Genus species ...")
        parts = rec.description.split()
        species = " ".join(parts[1:3]) if len(parts) >= 3 else parts[1] if len(parts) >= 2 else "Unknown"
        genus = parts[1] if len(parts) >= 2 else "Unknown"
        
        rows.append({
            "processid": rec.id,
            "nucleotides": str(rec.seq),
            "species_name": species,
            "genus_name": genus,
            "family_name": "",  # Not in FASTA headers
            "order_name": "",
            "bin_uri": "",
            "source": "GenBank"
        })
    return pd.DataFrame(rows)

In [ ]:
# Standardize all sources
dfs = []

if len(bold_aus) > 0:
    dfs.append(standardize_bold(bold_aus))
    print(f"BOLD Australia: {len(bold_aus)} records")

if len(bold_global) > 0:
    dfs.append(standardize_bold(bold_global))
    print(f"BOLD Global families: {len(bold_global)} records")

if genbank_records:
    gb_df = standardize_genbank(genbank_records)
    dfs.append(gb_df)
    print(f"GenBank: {len(gb_df)} records")

# Merge
merged = pd.concat(dfs, ignore_index=True)

# Remove rows without sequences
merged = merged.dropna(subset=["nucleotides"])
merged = merged[merged["nucleotides"].str.len() > 0]

# Deduplicate by sequence (keep first occurrence)
before_dedup = len(merged)
merged = merged.drop_duplicates(subset=["nucleotides"], keep="first")
print(f"\nDeduplication: {before_dedup} -> {len(merged)} records")

# Save
output_path = RAW_DIR / "merged_barcodes.csv"
merged.to_csv(output_path, index=False)
print(f"\nSaved merged dataset to {output_path}")

In [ ]:
# Summary statistics
print("=" * 60)
print("MERGED DATASET SUMMARY")
print("=" * 60)
print(f"Total sequences:  {len(merged):,}")
print(f"Unique species:   {merged['species_name'].nunique():,}")
print(f"Unique genera:    {merged['genus_name'].nunique():,}")
print(f"Unique families:  {merged['family_name'].nunique():,}")
print(f"\nSequence length stats:")
print(merged['nucleotides'].str.len().describe())
print(f"\nBy source:")
print(merged['source'].value_counts())
print("=" * 60)

## Next

Proceed to `02_data_cleaning.ipynb` for QC, taxonomy validation, and train/test splits.